# SSM local — l'équation d'intégration temporelle sous chaque nœud

On implémente le **SSM local** : chaque nœud/tuyau du Physarum porte une **mémoire récurrente** $h_t$ évoluant selon :

$$h_t = (1 - \Delta_t) \cdot h_{t-1} + \Delta_t \cdot x_t$$

La **surprise** $S_t$ du Predictive Coding pilote la constante de temps :

$$\Delta_t = \sigma(S_t) = \sigma(\Vert z_t - \hat{z}_t \Vert)$$

- $S \approx 0$ (prévisible) → $\Delta \to 0$ : le SSM **garde sa mémoire** (inertie, contexte stable).
- $S \gg 0$ (rupture) → $\Delta \to 1$ : la surprise **réinitialise** la mémoire (capte la nouveauté).

**Fusion espace + temps** : Physarum = espace (corrélations spatiales) ; Micro-SSM = temps (inertie) ; Surprise = vitesse du temps.

## 0. Imports

In [1]:
# SSM local — l'intégration temporelle sous chaque nœud
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from recherche_agi import (load_mnist, train_readout, LocalSSM, SSMLayer,
    surprise_to_delta, SSMNeuromodulatedReservoir, SynapticReservoir)

## 1. Données : MNIST

In [2]:
train_set, test_set = load_mnist()
print("Train :", len(train_set), "| Test :", len(test_set))

Train : 60000 | Test : 10000


## 2. Δ = σ(S) : la surprise pilote la constante de temps

In [3]:
print("=== Δ = σ(S) ===")
Ss = [0.0, 0.1, 0.5, 1.0, 2.0, 3.0]
for S in Ss:
    print(f"  S={S:.1f} → Δ={surprise_to_delta(S):.3f}")
print("\n  S≈0 → Δ≈0.5 (mémoire gardée) ; S≫0 → Δ→1 (mémoire réinitialisée)")

=== Δ = σ(S) ===
  S=0.0 → Δ=0.500
  S=0.1 → Δ=0.574
  S=0.5 → Δ=0.818
  S=1.0 → Δ=0.953
  S=2.0 → Δ=0.998
  S=3.0 → Δ=1.000

  S≈0 → Δ≈0.5 (mémoire gardée) ; S≫0 → Δ→1 (mémoire réinitialisée)


## 3. Comportement mémoire du SSM local

In [4]:
# S faible : la mémoire s'accumule lentement (inertie)
print("=== Mémoire avec surprise faible (Δ≈0.5) ===")
ssm = LocalSSM(1)
d_low = surprise_to_delta(0.05)
for t in range(5):
    h = ssm.step(np.array([1.0]), d_low)
print(f"  x=1 pendant 5 pas : h={h[0]:.4f} (accumulation lente)")

# S forte : la mémoire capture instantanément
print("\n=== Mémoire avec surprise forte (Δ≈1) ===")
ssm.reset()
d_high = surprise_to_delta(3.0)
for t in range(5):
    h = ssm.step(np.array([1.0]), d_high)
print(f"  x=1 pendant 5 pas : h={h[0]:.4f} (capture rapide)")

# Rupture : la surprise réinitialise
print("\n=== Rupture : la surprise réinitialise la mémoire ===")
ssm.reset()
for _ in range(3):
    ssm.step(np.array([0.0]), surprise_to_delta(0.1))
h_before = ssm.state[0]
h_after = ssm.step(np.array([10.0]), surprise_to_delta(3.0))[0]
print(f"  Avant saut : h={h_before:.3f} | après saut (S forte) : h={h_after:.3f}")

=== Mémoire avec surprise faible (Δ≈0.5) ===
  x=1 pendant 5 pas : h=0.9788 (accumulation lente)

=== Mémoire avec surprise forte (Δ≈1) ===
  x=1 pendant 5 pas : h=1.0000 (capture rapide)

=== Rupture : la surprise réinitialise la mémoire ===
  Avant saut : h=0.000 | après saut (S forte) : h=9.999


## 4. Intégration : SSM + réservoir neuromodulé (fusion espace/temps)

Le `SSMNeuromodulatedReservoir` combine Physarum (espace) + SSM (temps) : à chaque image, la surprise S pilote Δ, le SSM intègre z dans une mémoire h_t, et la signature = **[h_t, z]** (mémoire + nouveauté).

In [5]:
def extract(reservoir, dataset, n):
    X, y = [], []
    cnt = [0]*10
    for i in range(len(dataset)):
        l = int(dataset[i][1])
        if cnt[l] >= n//10: continue
        reservoir.reset_memory()
        X.append(reservoir.signature(dataset[i][0].squeeze().numpy()))
        y.append(l); cnt[l] += 1
        if sum(cnt) >= n: break
    return np.array(X), np.array(y)

# prototypes pour le prédicteur (sur signatures de base)
base = SynapticReservoir(alpha=5.0, n_iter=10, n_zones=64, downscale=8, eta=0.1, gamma=0.1, beta=5.0)
def extract_base(reservoir, dataset, n):
    X, y = [], []
    cnt = [0]*10
    for i in range(len(dataset)):
        l = int(dataset[i][1])
        if cnt[l] >= n//10: continue
        X.append(reservoir.signature(dataset[i][0].squeeze().numpy()))
        y.append(l); cnt[l] += 1
        if sum(cnt) >= n: break
    return np.array(X), np.array(y)
X0, y0 = extract_base(base, train_set, 200)
protos = {d: X0[y0==d].mean(axis=0) for d in range(10)}
def predictor(z):
    return min(protos.values(), key=lambda p: np.linalg.norm(p - z))

# réservoir avec mémoire temporelle SSM
ssm_res = SSMNeuromodulatedReservoir(axes=('top_down','left_right'), n_zones=32, downscale=8,
                                     predictor=predictor, n_max=15)
Xtr, ytr = extract(ssm_res, train_set, 150)
print(f"Signatures [mémoire h_t, nouveauté z] : {Xtr.shape}")
print(f"  surprise={ssm_res.last_surprise:.3f} → Δ={ssm_res.last_delta:.3f}")

Signatures [mémoire h_t, nouveauté z] : (150, 128)
  surprise=1.345 → Δ=0.983


In [6]:
# Couche lue sur les signatures mémoire-temporelles
ro = train_readout(Xtr, ytr, n_classes=10, epochs=50)
with torch.no_grad():
    acc = (ro(torch.tensor(Xtr, dtype=torch.float32)).argmax(1) == torch.tensor(ytr)).float().mean().item()
print(f"Acc couche lue (mémoire SSM) : {acc:.3f}")

Acc couche lue (mémoire SSM) : 0.467


## 5. Synthèse

In [7]:
print("=== SYNTHÈSE : SSM LOCAL ===")
print("Le SSM local sous chaque nœud donne au réservoir une MÉMOIRE TEMPORELLE :")
print("  h_t = (1-Δ)·h_{t-1} + Δ·x_t, avec Δ = σ(S)")
print()
print("  - S≈0 → Δ≈0 : inertie (contexte stable, prévisible)")
print("  - S≫0 → Δ≈1 : réinitialisation (rupture, nouveauté)")
print()
print("La surprise du Predictive Coding pilote la VITESSE DU TEMPS local,")
print("fusionnant espace (Physarum) et temps (SSM) en une signature [h_t, z].")
print(f"Acc couche lue : {acc:.3f}")

=== SYNTHÈSE : SSM LOCAL ===
Le SSM local sous chaque nœud donne au réservoir une MÉMOIRE TEMPORELLE :
  h_t = (1-Δ)·h_{t-1} + Δ·x_t, avec Δ = σ(S)

  - S≈0 → Δ≈0 : inertie (contexte stable, prévisible)
  - S≫0 → Δ≈1 : réinitialisation (rupture, nouveauté)

La surprise du Predictive Coding pilote la VITESSE DU TEMPS local,
fusionnant espace (Physarum) et temps (SSM) en une signature [h_t, z].
Acc couche lue : 0.467
